
# Этап 3: фиксированная SVM-классификация

Этот notebook не повторяет ERP/TFR-screening. Он делает только классификационный ablation:

- `selected` = `best_ch_by_power[PATIENT_ID]`;
- `rejected` = `ch_to_keep[PATIENT_ID] - selected`;
- одинаковые принятые эпохи;
- одинаковый preprocessing и CAR;
- одинаковые CV-folds;
- одна фиксированная SVM;
- полный перебор заранее заданных временных и частотных окон.

Результат: macro-F1, accuracy, balanced accuracy и ROC-AUC для selected/rejected по всей замороженной сетке.


## 0. Настройки

In [2]:

from pathlib import Path

PATIENT_ID = "s11"

CONFIG_PATH = Path(
    "/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/config.py"
)
STUDY_DEF_PATH = Path(
    "/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/"
    "study_definition_open_vs_all"
)
DATA_ROOT = Path(
    "/trinity/home/t.samsonov/notebooks/Pirogov/PirogovDATA"
)
OUTPUT_STORAGE = Path(
    "/trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/"
    "stage3_fixed_svm_ablation"
)

SESSIONS_TO_USE = None

# Fixed model. Do not change between ablation conditions.
SVM_KERNEL = "linear"
SVM_C = 752.6521969107142
SVM_GAMMA = "scale"
SVM_CLASS_WEIGHT = None

CV_N_SPLITS = 5
CV_SEED = 42

NOTCH_FREQS = (50.0, 100.0, 150.0)
L_FREQ = 0.1
H_FREQ = 120.0

# Time is relative to the LSL marker.
EPOCH_TMIN = -0.9
EPOCH_TMAX = 2.1
BASELINE = (-0.1, 0.0)

TFR_FMIN = 0.1
TFR_FMAX = 120.0
TFR_N_FREQS = 100
TFR_DECIM = 20
TFR_EPOCH_BATCH_SIZE = 24
TFR_N_JOBS = 1
LOG_POWER_EPS = 1e-20

TIME_WINDOWS = {
    "T0_marker_0_1": (0.0, 1.0),
    "T1_robot_0_1": (0.1, 1.1),
    "T2_robot_0_0p5": (0.1, 0.6),
    "T3_robot_0_1p5": (0.1, 1.6),
    "T4_robot_0_2": (0.1, 2.1),
}

FREQUENCY_BANDS = {
    "F0_current_0p1_59p4": (0.1, 59.4),
    "F1_low_0p1_30": (0.1, 30.0),
    "F2_alpha_beta_8_30": (8.0, 30.0),
    "F3_low_gamma_30_59p4": (30.0, 59.4),
    "F4_full_0p1_120": (0.1, 120.0),
    "F5_upper_59p4_120": (59.4, 120.0),
}

BASELINE_TIME_ID = "T0_marker_0_1"
BASELINE_FREQ_ID = "F0_current_0p1_59p4"

REUSE_FEATURE_CACHE = True


## 1. Импорты и функции

In [3]:

from __future__ import annotations

import importlib.util
import io
import json
import platform
import sys
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from typing import Any

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 180


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_figure(fig: plt.Figure, path_without_suffix: Path) -> None:
    ensure_dir(path_without_suffix.parent)
    fig.savefig(path_without_suffix.with_suffix(".png"), bbox_inches="tight")
    fig.savefig(path_without_suffix.with_suffix(".svg"), bbox_inches="tight")
    plt.close(fig)


def load_external_config(path: Path):
    path = path.expanduser().resolve()
    spec = importlib.util.spec_from_file_location(
        "stage3_external_config",
        str(path),
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load config: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def load_manifest(root: Path) -> pd.DataFrame:
    root = root.expanduser()

    if root.is_dir():
        direct = root / "sample_manifest.csv"
        if direct.exists():
            return pd.read_csv(direct)

        matches = list(root.rglob("sample_manifest.csv"))
        if len(matches) == 1:
            return pd.read_csv(matches[0])

        raise FileNotFoundError(
            f"sample_manifest.csv not found under {root}"
        )

    if root.suffix.lower() == ".zip":
        with zipfile.ZipFile(root, "r") as zf:
            matches = [
                name for name in zf.namelist()
                if name == "sample_manifest.csv"
                or name.endswith("/sample_manifest.csv")
            ]
            if len(matches) != 1:
                raise FileNotFoundError(
                    f"Expected one sample_manifest.csv, found {matches}"
                )
            with zf.open(matches[0]) as stream:
                return pd.read_csv(stream)

    raise ValueError(f"Unsupported STUDY_DEF_PATH: {root}")


def parse_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )


def resolve_edf_path(row: pd.Series) -> Path:
    original = Path(str(row["edf_path"]))
    if original.exists():
        return original

    session_dir = DATA_ROOT / str(row["session"])
    candidate = session_dir / original.name
    if candidate.exists():
        return candidate

    matches = sorted(session_dir.glob("*.edf"))
    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"EDF not found for {row['session']}. "
        f"Original={original}; candidate={candidate}; matches={matches}"
    )


def resolve_channel_names(
    requested: list[str],
    available: list[str],
) -> tuple[list[str], list[str]]:
    actual_by_lower = {name.lower(): name for name in available}
    resolved: list[str] = []
    missing: list[str] = []

    for name in requested:
        actual = actual_by_lower.get(name.lower())
        if actual is None:
            missing.append(name)
        elif actual not in resolved:
            resolved.append(actual)

    return resolved, missing


def preprocess_raw(
    raw: mne.io.BaseRaw,
    candidate_channels: list[str],
) -> mne.io.BaseRaw:
    """
    Fixed preprocessing for every condition.

    CAR is calculated once across all ch_to_keep channels before the
    selected/rejected split.
    """
    out = raw.copy().pick(candidate_channels).load_data()
    nyquist = float(out.info["sfreq"]) / 2.0

    for freq in NOTCH_FREQS:
        if freq < nyquist:
            out.notch_filter(
                freqs=[freq],
                method="iir",
                iir_params={"order": 2, "ftype": "butter"},
                verbose=False,
            )

    out.filter(
        l_freq=L_FREQ,
        h_freq=min(H_FREQ, nyquist - 1e-6),
        method="iir",
        iir_params={"order": 4, "ftype": "butter"},
        verbose=False,
    )

    data = out.get_data()
    data -= data.mean(axis=0, keepdims=True)

    car = mne.io.RawArray(
        data,
        out.info.copy(),
        verbose=False,
    )
    car.set_annotations(out.annotations.copy())
    return car


def make_session_epochs(
    rows: pd.DataFrame,
    configured_channels: list[str],
) -> tuple[mne.Epochs, dict[str, Any]]:
    rows = rows.sort_values(
        ["sample", "annotation_index"]
    ).reset_index(drop=True)

    edf_path = resolve_edf_path(rows.iloc[0])
    raw = mne.io.read_raw_edf(
        edf_path,
        preload=True,
        verbose=False,
    )

    actual_channels, missing = resolve_channel_names(
        configured_channels,
        raw.ch_names,
    )
    if missing:
        warnings.warn(
            f"{rows['session'].iloc[0]}: missing channels {missing}"
        )
    if not actual_channels:
        raise RuntimeError("No configured channels are present in EDF.")

    raw = preprocess_raw(raw, actual_channels)

    events = np.zeros((len(rows), 3), dtype=int)
    events[:, 0] = rows["sample"].astype(int).to_numpy()
    events[:, 2] = rows["class_label"].astype(int).to_numpy() + 1

    metadata_columns = [
        "epoch_id",
        "session",
        "annotation_index",
        "event_code",
        "gesture_name",
        "class_name",
        "class_label",
        "sample",
        "marker_onset_s",
        "robot_onset_s",
    ]
    metadata = rows[metadata_columns].copy()

    epochs = mne.Epochs(
        raw,
        events,
        event_id={
            "all_other_gestures": 1,
            "open_hand": 2,
        },
        tmin=EPOCH_TMIN,
        tmax=EPOCH_TMAX,
        baseline=BASELINE,
        preload=True,
        reject=None,
        metadata=metadata,
        event_repeated="drop",
        verbose=False,
    )

    audit = {
        "session": str(rows["session"].iloc[0]),
        "edf_path": str(edf_path),
        "input_rows": len(rows),
        "retained_epochs": len(epochs),
        "actual_channels": "|".join(actual_channels),
        "missing_channels": "|".join(missing),
    }
    return epochs, audit


def make_splits(
    labels: np.ndarray,
) -> tuple[list[tuple[np.ndarray, np.ndarray]], np.ndarray]:
    cv = StratifiedKFold(
        n_splits=CV_N_SPLITS,
        shuffle=True,
        random_state=CV_SEED,
    )
    splits: list[tuple[np.ndarray, np.ndarray]] = []
    fold_assignment = np.full(len(labels), -1, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(np.zeros(len(labels)), labels)
    ):
        splits.append((train_idx, test_idx))
        fold_assignment[test_idx] = fold

    if np.any(fold_assignment < 0):
        raise RuntimeError("Incomplete fold assignment.")

    return splits, fold_assignment


def make_model() -> Pipeline:
    svc_kwargs: dict[str, Any] = {
        "C": float(SVM_C),
        "kernel": str(SVM_KERNEL),
        "class_weight": SVM_CLASS_WEIGHT,
        "probability": False,
        "random_state": CV_SEED,
    }
    if SVM_KERNEL in {"rbf", "poly", "sigmoid"}:
        svc_kwargs["gamma"] = SVM_GAMMA

    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("svc", SVC(**svc_kwargs)),
        ]
    )


@dataclass(frozen=True)
class Condition:
    channel_group: str
    time_id: str
    freq_id: str
    channels: tuple[str, ...]
    time_start: float
    time_end: float
    freq_start: float
    freq_end: float


## 2. Формирование selected и rejected

In [4]:

cfg = load_external_config(CONFIG_PATH)

all_configured = list(cfg.ch_to_keep[PATIENT_ID])
selected_configured = list(cfg.best_ch_by_power[PATIENT_ID])

selected_lower = {name.lower() for name in selected_configured}
rejected_configured = [
    name
    for name in all_configured
    if name.lower() not in selected_lower
]

if not selected_configured:
    raise RuntimeError(
        f"best_ch_by_power[{PATIENT_ID}] is empty."
    )
if not rejected_configured:
    raise RuntimeError(
        f"No rejected channels for {PATIENT_ID}."
    )

print("Patient:", PATIENT_ID)
print("All:", all_configured)
print("Selected:", selected_configured)
print("Rejected:", rejected_configured)


Patient: s11
All: ['Fp1', 'F7', 'Ft7', 'T3', 'F8', 'Fp2', 'Fpz', 'F3', 'Fc3', 'Tp7', 'Fc4']
Selected: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Rejected: ['Ft7', 'T3', 'Fc3', 'Fc4']


## 3. Принятые эпохи и выходные папки

In [5]:

manifest = load_manifest(STUDY_DEF_PATH)

manifest["accepted_after_rejection"] = parse_bool(
    manifest["accepted_after_rejection"]
)
manifest["included_in_main_task"] = parse_bool(
    manifest["included_in_main_task"]
)

patient_rows = manifest[
    (manifest["subject"] == PATIENT_ID)
    & (manifest["accepted_after_rejection"] == True)
    & (manifest["included_in_main_task"] == True)
    & (manifest["class_label"].isin([0, 1]))
].copy()

if SESSIONS_TO_USE is not None:
    patient_rows = patient_rows[
        patient_rows["session"].isin(SESSIONS_TO_USE)
    ].copy()

if patient_rows.empty:
    raise RuntimeError(
        f"No accepted task epochs for {PATIENT_ID}."
    )

session_names = sorted(patient_rows["session"].unique())

RUN_ROOT = ensure_dir(OUTPUT_STORAGE / PATIENT_ID)
DIRS = {
    "run_info": ensure_dir(RUN_ROOT / "00_run_info"),
    "data_audit": ensure_dir(RUN_ROOT / "01_data_audit"),
    "selected": ensure_dir(RUN_ROOT / "02_selected_channels"),
    "rejected": ensure_dir(RUN_ROOT / "03_rejected_channels"),
    "comparison": ensure_dir(
        RUN_ROOT / "04_selected_vs_rejected"
    ),
    "predictions": ensure_dir(RUN_ROOT / "05_predictions"),
    "cache": ensure_dir(RUN_ROOT / "_cache"),
}

print("Sessions:", session_names)
print("Output:", RUN_ROOT)


Sessions: ['2025-09-05_s11/session_1', '2025-09-05_s11/session_2']
Output: /trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/stage3_fixed_svm_ablation/s11


## 4. Загрузка EDF и общий preprocessing

In [6]:

session_epochs: list[mne.Epochs] = []
audit_rows: list[dict[str, Any]] = []

for session_name in session_names:
    rows = patient_rows[
        patient_rows["session"] == session_name
    ].copy()

    session_ep, audit = make_session_epochs(
        rows,
        all_configured,
    )
    session_epochs.append(session_ep)
    audit_rows.append(audit)
    print(audit)

orders = [tuple(ep.ch_names) for ep in session_epochs]
if len(set(orders)) != 1:
    raise RuntimeError(
        f"Channel order differs between sessions: {orders}"
    )

epochs = mne.concatenate_epochs(
    session_epochs,
    add_offset=True,
    verbose=False,
)

actual_all = list(epochs.ch_names)
actual_selected, missing_selected = resolve_channel_names(
    selected_configured,
    actual_all,
)
actual_rejected, missing_rejected = resolve_channel_names(
    rejected_configured,
    actual_all,
)

if missing_selected or missing_rejected:
    warnings.warn(
        f"Missing after resolution: selected={missing_selected}, "
        f"rejected={missing_rejected}"
    )

if not actual_selected or not actual_rejected:
    raise RuntimeError(
        f"Invalid groups: selected={actual_selected}, "
        f"rejected={actual_rejected}"
    )

metadata = epochs.metadata.reset_index(drop=True).copy()
y = metadata["class_label"].astype(int).to_numpy()

pd.DataFrame(audit_rows).to_csv(
    DIRS["data_audit"] / "session_loading_audit.csv",
    index=False,
)
metadata.to_csv(
    DIRS["data_audit"] / "epoch_metadata.csv",
    index=False,
)

print("Epochs:", len(epochs))
print("Class counts:", pd.Series(y).value_counts().to_dict())
print("Actual selected:", actual_selected)
print("Actual rejected:", actual_rejected)


{'session': '2025-09-05_s11/session_1', 'edf_path': '/beegfs/home/t.samsonov/notebooks/Pirogov/PirogovDATA/2025-09-05_s11/session_1/NeoRec_2025-09-05_21-04-18.edf', 'input_rows': 135, 'retained_epochs': 135, 'actual_channels': 'Fp1|F7|Ft7|T3|F8|Fp2|Fpz|F3|Fc3|Tp7|Fc4', 'missing_channels': ''}
{'session': '2025-09-05_s11/session_2', 'edf_path': '/beegfs/home/t.samsonov/notebooks/Pirogov/PirogovDATA/2025-09-05_s11/session_2/NeoRec_2025-09-05_21-19-05.edf', 'input_rows': 400, 'retained_epochs': 400, 'actual_channels': 'Fp1|F7|Ft7|T3|F8|Fp2|Fpz|F3|Fc3|Tp7|Fc4', 'missing_channels': ''}
Epochs: 535
Class counts: {1: 268, 0: 267}
Actual selected: ['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'F8', 'Tp7']
Actual rejected: ['Ft7', 'T3', 'Fc3', 'Fc4']


/tmp/ipykernel_2608936/3308715939.py:23: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs = mne.concatenate_epochs(


## 5. Одни и те же CV-folds для всех условий

In [7]:

splits, fold_assignment = make_splits(y)

fold_table = metadata.copy()
fold_table["epoch_index"] = np.arange(len(metadata))
fold_table["validation_fold"] = fold_assignment

fold_table.to_csv(
    DIRS["run_info"] / "fixed_cv_fold_assignment.csv",
    index=False,
)

display(
    fold_table
    .groupby(["validation_fold", "class_name"])
    .size()
    .unstack(fill_value=0)
)


class_name,all_other_gestures,open_hand
validation_fold,,
0,54,53
1,54,53
2,53,54
3,53,54
4,53,54



## 6. Вычисление признаков

Для экономии памяти полный TFR не сохраняется. Morlet power считается пакетами,
после чего для каждого временного окна немедленно вычисляется средняя мощность.

Признак одного условия:

```text
log10(mean power over selected time window)
→ выбрать channel group
→ выбрать frequency band
→ flatten(channel × frequency)
→ StandardScaler внутри train-fold
→ fixed SVM
```


In [8]:

freqs = np.linspace(
    TFR_FMIN,
    TFR_FMAX,
    TFR_N_FREQS,
)
n_cycles = freqs / 2.0
tfr_times = epochs.times[::TFR_DECIM]

time_masks: dict[str, np.ndarray] = {}
for time_id, (start, stop) in TIME_WINDOWS.items():
    mask = (tfr_times >= start) & (tfr_times <= stop)
    if not np.any(mask):
        raise RuntimeError(
            f"No TFR points for {time_id}: {start}..{stop}"
        )
    time_masks[time_id] = mask

cache_path = DIRS["cache"] / (
    f"{PATIENT_ID}__time_pooled_log_power__"
    f"{TFR_N_FREQS}freq__decim{TFR_DECIM}.npz"
)

if REUSE_FEATURE_CACHE and cache_path.exists():
    print("Loading cache:", cache_path)
    cache = np.load(cache_path, allow_pickle=False)

    cache_channels = list(cache["channels"].astype(str))
    if cache_channels != actual_all:
        raise RuntimeError(
            f"Cache channels differ: {cache_channels} != {actual_all}"
        )
    if not np.allclose(cache["freqs"], freqs):
        raise RuntimeError("Cache frequencies differ.")

    time_features = {
        time_id: cache[f"features__{time_id}"]
        for time_id in TIME_WINDOWS
    }

else:
    n_epochs = len(epochs)
    n_channels = len(actual_all)

    time_features = {
        time_id: np.empty(
            (n_epochs, n_channels, len(freqs)),
            dtype=np.float32,
        )
        for time_id in TIME_WINDOWS
    }

    extraction_start = time.time()

    for channel_index, channel_name in enumerate(actual_all):
        print(
            f"{channel_index + 1}/{n_channels}: {channel_name}",
            flush=True,
        )

        channel_data = epochs.get_data(
            picks=[channel_name],
        ).astype(np.float32, copy=False)

        for batch_start in range(
            0,
            n_epochs,
            TFR_EPOCH_BATCH_SIZE,
        ):
            batch_stop = min(
                batch_start + TFR_EPOCH_BATCH_SIZE,
                n_epochs,
            )
            batch = channel_data[batch_start:batch_stop]

            power = mne.time_frequency.tfr_array_morlet(
                batch,
                sfreq=float(epochs.info["sfreq"]),
                freqs=freqs,
                n_cycles=n_cycles,
                output="power",
                decim=TFR_DECIM,
                n_jobs=TFR_N_JOBS,
                verbose=False,
            )[:, 0]

            for time_id, mask in time_masks.items():
                pooled = power[:, :, mask].mean(axis=-1)
                time_features[time_id][
                    batch_start:batch_stop,
                    channel_index,
                    :,
                ] = np.log10(
                    np.maximum(pooled, LOG_POWER_EPS)
                ).astype(np.float32)

            del power

    elapsed = (time.time() - extraction_start) / 60.0
    print(f"Feature extraction completed in {elapsed:.1f} min")

    payload = {
        "channels": np.asarray(actual_all),
        "freqs": freqs,
    }
    payload.update(
        {
            f"features__{time_id}": values
            for time_id, values in time_features.items()
        }
    )
    np.savez_compressed(cache_path, **payload)
    print("Saved cache:", cache_path)

for time_id, values in time_features.items():
    print(time_id, values.shape)


1/11: Fp1
2/11: F7
3/11: Ft7
4/11: T3
5/11: F8
6/11: Fp2
7/11: Fpz
8/11: F3
9/11: Fc3
10/11: Tp7
11/11: Fc4
Feature extraction completed in 1.6 min
Saved cache: /trinity/home/t.samsonov/notebooks/Pirogov/PreprocessedData/stage3_fixed_svm_ablation/s11/_cache/s11__time_pooled_log_power__100freq__decim20.npz
T0_marker_0_1 (535, 11, 100)
T1_robot_0_1 (535, 11, 100)
T2_robot_0_0p5 (535, 11, 100)
T3_robot_0_1p5 (535, 11, 100)
T4_robot_0_2 (535, 11, 100)


## 7. Замороженная сетка условий

In [9]:

channel_groups = {
    "selected": actual_selected,
    "rejected": actual_rejected,
}

conditions: list[Condition] = []

for group_name, group_channels in channel_groups.items():
    for time_id, (tmin, tmax) in TIME_WINDOWS.items():
        for freq_id, (fmin, fmax) in FREQUENCY_BANDS.items():
            conditions.append(
                Condition(
                    channel_group=group_name,
                    time_id=time_id,
                    freq_id=freq_id,
                    channels=tuple(group_channels),
                    time_start=tmin,
                    time_end=tmax,
                    freq_start=fmin,
                    freq_end=fmax,
                )
            )

condition_table = pd.DataFrame(
    [asdict(condition) for condition in conditions]
)
condition_table["channels"] = condition_table[
    "channels"
].apply(lambda values: "|".join(values))

condition_table.to_csv(
    DIRS["run_info"] / "frozen_ablation_conditions.csv",
    index=False,
)

print("Conditions:", len(conditions))
display(condition_table.head())


Conditions: 60


,channel_group,time_id,freq_id,channels,time_start,time_end,freq_start,freq_end
0,selected,T0_marker_0_1,F0_current_0p1_59p4,Fp1|Fpz|Fp2|F7|F3|F8|Tp7,0.0,1.0,0.1,59.4
1,selected,T0_marker_0_1,F1_low_0p1_30,Fp1|Fpz|Fp2|F7|F3|F8|Tp7,0.0,1.0,0.1,30.0
2,selected,T0_marker_0_1,F2_alpha_beta_8_30,Fp1|Fpz|Fp2|F7|F3|F8|Tp7,0.0,1.0,8.0,30.0
3,selected,T0_marker_0_1,F3_low_gamma_30_59p4,Fp1|Fpz|Fp2|F7|F3|F8|Tp7,0.0,1.0,30.0,59.4
4,selected,T0_marker_0_1,F4_full_0p1_120,Fp1|Fpz|Fp2|F7|F3|F8|Tp7,0.0,1.0,0.1,120.0


## 8. Классификация

In [10]:

def condition_matrix(condition: Condition) -> np.ndarray:
    channel_indices = [
        actual_all.index(channel)
        for channel in condition.channels
    ]
    frequency_mask = (
        (freqs >= condition.freq_start)
        & (freqs <= condition.freq_end)
    )

    tensor = time_features[condition.time_id][
        :,
        channel_indices,
        :,
    ][:, :, frequency_mask]

    return tensor.reshape(len(tensor), -1)


fold_rows: list[dict[str, Any]] = []
prediction_frames: list[pd.DataFrame] = []

classification_start = time.time()

for condition_number, condition in enumerate(conditions, start=1):
    X = condition_matrix(condition)

    print(
        f"{condition_number:02d}/{len(conditions)} | "
        f"{condition.channel_group} | "
        f"{condition.time_id} | {condition.freq_id} | "
        f"X={X.shape}",
        flush=True,
    )

    current_predictions: list[pd.DataFrame] = []

    for fold, (train_idx, test_idx) in enumerate(splits):
        model = make_model()
        model.fit(X[train_idx], y[train_idx])

        y_pred = model.predict(X[test_idx])
        decision = model.decision_function(X[test_idx])

        try:
            auc = float(
                roc_auc_score(y[test_idx], decision)
            )
        except ValueError:
            auc = float("nan")

        fold_rows.append(
            {
                **asdict(condition),
                "channels": "|".join(condition.channels),
                "n_channels": len(condition.channels),
                "n_features": X.shape[1],
                "fold": fold,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "accuracy": float(
                    accuracy_score(y[test_idx], y_pred)
                ),
                "balanced_accuracy": float(
                    balanced_accuracy_score(
                        y[test_idx],
                        y_pred,
                    )
                ),
                "macro_f1": float(
                    f1_score(
                        y[test_idx],
                        y_pred,
                        average="macro",
                    )
                ),
                "roc_auc": auc,
            }
        )

        fold_predictions = metadata.iloc[test_idx].copy()
        fold_predictions["epoch_index"] = test_idx
        fold_predictions["fold"] = fold
        fold_predictions["channel_group"] = (
            condition.channel_group
        )
        fold_predictions["time_id"] = condition.time_id
        fold_predictions["freq_id"] = condition.freq_id
        fold_predictions["y_true"] = y[test_idx]
        fold_predictions["y_pred"] = y_pred
        fold_predictions["decision_function"] = decision
        current_predictions.append(fold_predictions)

    prediction_frames.append(
        pd.concat(
            current_predictions,
            ignore_index=True,
        )
    )

fold_results = pd.DataFrame(fold_rows)
predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

fold_results.to_csv(
    RUN_ROOT / "all_fold_results.csv",
    index=False,
)
predictions.to_csv(
    DIRS["predictions"] / "all_out_of_fold_predictions.csv",
    index=False,
)

print(
    "Classification time:",
    f"{(time.time() - classification_start) / 60.0:.1f} min",
)


01/60 | selected | T0_marker_0_1 | F0_current_0p1_59p4 | X=(535, 343)
02/60 | selected | T0_marker_0_1 | F1_low_0p1_30 | X=(535, 175)
03/60 | selected | T0_marker_0_1 | F2_alpha_beta_8_30 | X=(535, 126)


## 9. Сводная таблица

In [ ]:

group_columns = [
    "channel_group",
    "time_id",
    "freq_id",
    "channels",
    "n_channels",
    "n_features",
    "time_start",
    "time_end",
    "freq_start",
    "freq_end",
]

summary = (
    fold_results
    .groupby(group_columns, as_index=False)
    .agg(
        macro_f1_median=("macro_f1", "median"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_median=("accuracy", "median"),
        balanced_accuracy_median=(
            "balanced_accuracy",
            "median",
        ),
        roc_auc_median=("roc_auc", "median"),
    )
)

q25 = (
    fold_results
    .groupby(group_columns)["macro_f1"]
    .quantile(0.25)
    .rename("macro_f1_q25")
    .reset_index()
)
q75 = (
    fold_results
    .groupby(group_columns)["macro_f1"]
    .quantile(0.75)
    .rename("macro_f1_q75")
    .reset_index()
)

summary = summary.merge(q25, on=group_columns)
summary = summary.merge(q75, on=group_columns)
summary = summary.sort_values(
    ["channel_group", "macro_f1_median"],
    ascending=[True, False],
).reset_index(drop=True)

summary.to_csv(
    RUN_ROOT / "condition_summary.csv",
    index=False,
)

summary[
    summary["channel_group"] == "selected"
].to_csv(
    DIRS["selected"] / "selected_condition_summary.csv",
    index=False,
)
summary[
    summary["channel_group"] == "rejected"
].to_csv(
    DIRS["rejected"] / "rejected_condition_summary.csv",
    index=False,
)

display(summary.head(12))


## 10. Heatmap для selected и rejected

In [ ]:

def plot_macro_f1_grid(
    group_name: str,
    output_dir: Path,
) -> None:
    group = summary[
        summary["channel_group"] == group_name
    ]

    matrix = group.pivot(
        index="time_id",
        columns="freq_id",
        values="macro_f1_median",
    ).reindex(
        index=list(TIME_WINDOWS),
        columns=list(FREQUENCY_BANDS),
    )

    fig, ax = plt.subplots(
        figsize=(12, 6),
        constrained_layout=True,
    )
    image = ax.imshow(
        matrix.to_numpy(),
        aspect="auto",
        interpolation="nearest",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )

    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(
        matrix.columns,
        rotation=35,
        ha="right",
    )
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Frequency band")
    ax.set_ylabel("Time window")
    ax.set_title(
        f"{PATIENT_ID} | {group_name} | median CV macro-F1\n"
        f"channels={channel_groups[group_name]}"
    )

    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            ax.text(
                column,
                row,
                f"{value:.3f}",
                ha="center",
                va="center",
                fontsize=9,
            )

    fig.colorbar(
        image,
        ax=ax,
        label="Median macro-F1",
    )
    save_figure(
        fig,
        output_dir / f"01_{group_name}_macro_f1_grid",
    )


plot_macro_f1_grid("selected", DIRS["selected"])
plot_macro_f1_grid("rejected", DIRS["rejected"])


## 11. Selected минус rejected

In [ ]:

main = summary[
    summary["channel_group"].isin(["selected", "rejected"])
]

comparison = main.pivot_table(
    index=["time_id", "freq_id"],
    columns="channel_group",
    values="macro_f1_median",
).reset_index()

comparison["selected_minus_rejected"] = (
    comparison["selected"] - comparison["rejected"]
)
comparison.to_csv(
    DIRS["comparison"] / "selected_vs_rejected.csv",
    index=False,
)

difference = comparison.pivot(
    index="time_id",
    columns="freq_id",
    values="selected_minus_rejected",
).reindex(
    index=list(TIME_WINDOWS),
    columns=list(FREQUENCY_BANDS),
)

limit = max(
    0.01,
    float(np.nanmax(np.abs(difference.to_numpy()))),
)

fig, ax = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)
image = ax.imshow(
    difference.to_numpy(),
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
)

ax.set_xticks(np.arange(len(difference.columns)))
ax.set_xticklabels(
    difference.columns,
    rotation=35,
    ha="right",
)
ax.set_yticks(np.arange(len(difference.index)))
ax.set_yticklabels(difference.index)
ax.set_xlabel("Frequency band")
ax.set_ylabel("Time window")
ax.set_title(
    f"{PATIENT_ID} | selected - rejected | median macro-F1"
)

for row in range(difference.shape[0]):
    for column in range(difference.shape[1]):
        value = difference.iloc[row, column]
        ax.text(
            column,
            row,
            f"{value:+.3f}",
            ha="center",
            va="center",
            fontsize=9,
        )

fig.colorbar(
    image,
    ax=ax,
    label="Delta median macro-F1",
)
save_figure(
    fig,
    DIRS["comparison"] / "01_selected_minus_rejected_grid",
)


## 12. Временной и частотный срезы

In [ ]:

def plot_slice(
    fixed_column: str,
    fixed_value: str,
    varying_column: str,
    varying_order: list[str],
    title: str,
    output_name: str,
) -> None:
    subset = main[
        main[fixed_column] == fixed_value
    ]

    fig, ax = plt.subplots(
        figsize=(12, 6),
        constrained_layout=True,
    )
    x = np.arange(len(varying_order))

    for group_name in ["selected", "rejected"]:
        group = (
            subset[
                subset["channel_group"] == group_name
            ]
            .set_index(varying_column)
            .reindex(varying_order)
        )

        ax.plot(
            x,
            group["macro_f1_median"],
            marker="o",
            linewidth=2,
            label=group_name,
        )
        ax.fill_between(
            x,
            group["macro_f1_q25"].astype(float),
            group["macro_f1_q75"].astype(float),
            alpha=0.15,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        varying_order,
        rotation=35,
        ha="right",
    )
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Macro-F1")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    save_figure(
        fig,
        DIRS["comparison"] / output_name,
    )


plot_slice(
    fixed_column="freq_id",
    fixed_value=BASELINE_FREQ_ID,
    varying_column="time_id",
    varying_order=list(TIME_WINDOWS),
    title=(
        f"{PATIENT_ID} | time-window ablation | "
        f"{BASELINE_FREQ_ID}"
    ),
    output_name="02_time_window_slice",
)

plot_slice(
    fixed_column="time_id",
    fixed_value=BASELINE_TIME_ID,
    varying_column="freq_id",
    varying_order=list(FREQUENCY_BANDS),
    title=(
        f"{PATIENT_ID} | frequency-band ablation | "
        f"{BASELINE_TIME_ID}"
    ),
    output_name="03_frequency_band_slice",
)


## 13. Baseline, best и confusion matrices

In [ ]:

baseline_rows = summary[
    (summary["time_id"] == BASELINE_TIME_ID)
    & (summary["freq_id"] == BASELINE_FREQ_ID)
].copy()
baseline_rows["role"] = "baseline"

best_rows = (
    summary
    .sort_values(
        ["channel_group", "macro_f1_median"],
        ascending=[True, False],
    )
    .groupby("channel_group", as_index=False)
    .head(1)
    .copy()
)
best_rows["role"] = "best_in_frozen_grid"

report = pd.concat(
    [baseline_rows, best_rows],
    ignore_index=True,
)
report.to_csv(
    DIRS["comparison"] / "baseline_and_best.csv",
    index=False,
)

display(
    report[
        [
            "role",
            "channel_group",
            "time_id",
            "freq_id",
            "n_features",
            "macro_f1_median",
            "accuracy_median",
            "balanced_accuracy_median",
            "roc_auc_median",
        ]
    ]
)


def plot_confusion(row: pd.Series) -> None:
    subset = predictions[
        (predictions["channel_group"] == row["channel_group"])
        & (predictions["time_id"] == row["time_id"])
        & (predictions["freq_id"] == row["freq_id"])
    ]

    matrix = confusion_matrix(
        subset["y_true"],
        subset["y_pred"],
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(5, 5),
        constrained_layout=True,
    )
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["all_other", "open_hand"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["all_other", "open_hand"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(
        f"{PATIENT_ID} | {row['channel_group']} | {row['role']}\n"
        f"{row['time_id']} | {row['freq_id']}"
    )

    for true_index in range(2):
        for pred_index in range(2):
            ax.text(
                pred_index,
                true_index,
                str(matrix[true_index, pred_index]),
                ha="center",
                va="center",
                fontsize=14,
            )

    fig.colorbar(image, ax=ax)
    save_figure(
        fig,
        DIRS["comparison"] / (
            f"confusion__{row['role']}__"
            f"{row['channel_group']}__"
            f"{row['time_id']}__{row['freq_id']}"
        ),
    )


for _, report_row in report.iterrows():
    plot_confusion(report_row)


## 14. Автоматическая текстовая сводка

In [ ]:

def one_row(
    group_name: str,
    time_id: str,
    freq_id: str,
) -> pd.Series:
    rows = summary[
        (summary["channel_group"] == group_name)
        & (summary["time_id"] == time_id)
        & (summary["freq_id"] == freq_id)
    ]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected one row: {group_name}/{time_id}/{freq_id}"
        )
    return rows.iloc[0]


selected_baseline = one_row(
    "selected",
    BASELINE_TIME_ID,
    BASELINE_FREQ_ID,
)
rejected_baseline = one_row(
    "rejected",
    BASELINE_TIME_ID,
    BASELINE_FREQ_ID,
)
selected_best = best_rows[
    best_rows["channel_group"] == "selected"
].iloc[0]
rejected_best = best_rows[
    best_rows["channel_group"] == "rejected"
].iloc[0]

summary_text = f"""
Patient: {PATIENT_ID}

Fixed experiment:
- accepted epochs: {len(epochs)}
- CV: {CV_N_SPLITS}-fold StratifiedKFold, seed={CV_SEED}
- model: StandardScaler(train only) + SVC(
    kernel={SVM_KERNEL}, C={SVM_C}, gamma={SVM_GAMMA}
  )
- CAR reference: all ch_to_keep channels before selected/rejected split

Channels:
- selected ({len(actual_selected)}): {actual_selected}
- rejected ({len(actual_rejected)}): {actual_rejected}

Baseline {BASELINE_TIME_ID} / {BASELINE_FREQ_ID}:
- selected macro-F1 median:
  {selected_baseline['macro_f1_median']:.4f}
- rejected macro-F1 median:
  {rejected_baseline['macro_f1_median']:.4f}
- selected - rejected:
  {selected_baseline['macro_f1_median'] - rejected_baseline['macro_f1_median']:+.4f}

Best selected in frozen grid:
- {selected_best['time_id']} / {selected_best['freq_id']}
- macro-F1 median={selected_best['macro_f1_median']:.4f}

Best rejected in frozen grid:
- {rejected_best['time_id']} / {rejected_best['freq_id']}
- macro-F1 median={rejected_best['macro_f1_median']:.4f}

Open these files:
1. 02_selected_channels/01_selected_macro_f1_grid.png
2. 03_rejected_channels/01_rejected_macro_f1_grid.png
3. 04_selected_vs_rejected/01_selected_minus_rejected_grid.png
4. 04_selected_vs_rejected/02_time_window_slice.png
5. 04_selected_vs_rejected/03_frequency_band_slice.png
6. 04_selected_vs_rejected/baseline_and_best.csv
7. all_fold_results.csv
""".strip()

(DIRS["run_info"] / "RESULT_SUMMARY.txt").write_text(
    summary_text,
    encoding="utf-8",
)
print(summary_text)


## 15. Сохранение конфигурации и версий

In [ ]:

run_config = {
    "patient_id": PATIENT_ID,
    "config_path": str(CONFIG_PATH),
    "study_definition": str(STUDY_DEF_PATH),
    "data_root": str(DATA_ROOT),
    "output_root": str(RUN_ROOT),
    "sessions": session_names,
    "all_channels_configured": all_configured,
    "selected_channels_configured": selected_configured,
    "rejected_channels_configured": rejected_configured,
    "all_channels_actual": actual_all,
    "selected_channels_actual": actual_selected,
    "rejected_channels_actual": actual_rejected,
    "n_epochs": len(epochs),
    "class_counts": {
        str(key): int(value)
        for key, value in pd.Series(y).value_counts().items()
    },
    "time_windows": TIME_WINDOWS,
    "frequency_bands": FREQUENCY_BANDS,
    "model": {
        "feature": "log10(mean Morlet power over time)",
        "flatten": True,
        "scaler": "StandardScaler fitted on each train fold",
        "kernel": SVM_KERNEL,
        "C": SVM_C,
        "gamma": SVM_GAMMA,
        "class_weight": SVM_CLASS_WEIGHT,
    },
    "cv": {
        "type": "StratifiedKFold",
        "n_splits": CV_N_SPLITS,
        "seed": CV_SEED,
    },
    "versions": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "mne": mne.__version__,
        "scikit_learn": sklearn.__version__,
    },
}

(DIRS["run_info"] / "run_config.json").write_text(
    json.dumps(
        run_config,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print(json.dumps(run_config["versions"], indent=2))
